# California Housing Dataset

Implementación con PyTorch.

In [9]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.optim as optim


from torch import nn
from torch.utils.data import random_split, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

# Obteniendo el dataset y partición

In [11]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

# Clases y funciones usadas

In [ ]:
def compute_params(X_train, feature_mask):

    # A tensor
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    mask = feature_mask
    X_selected = X_train_tensor[:, mask]

    # Scaling
    means = X_selected.mean(dim=0)
    stds = X_selected.std(dim=0)

    Q1 = torch.quantile(X_selected, 0.25, dim=0)
    Q3 = torch.quantile(X_selected, 0.75, dim=0)
    IQR = Q3 - Q1

    # Bounds para el trimming
    lower_bounds = Q1 - 1.5 * IQR
    upper_bounds = Q3 + 1.5 * IQR

    return means, stds, lower_bounds, upper_bounds

In [ ]:
class ScalingLayer(nn.Module):

    def __init__(self, mean: torch.Tensor, std: torch.Tensor, feature_mask: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
        self.register_buffer('feature_mask', feature_mask)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
      # Para no modificar el original
        X_scaled = X.clone()

        mask = self.feature_mask

        X_scaled[:, mask] = (X_scaled[:, mask] - self.mean) / (self.std + 1e-8)

        return X_scaled

In [ ]:
class OutlierTrimmingLayer(nn.Module):

    def __init__(self, lower_bounds: torch.Tensor, upper_bounds: torch.Tensor, feature_mask: torch.Tensor):
        super().__init__()
        self.register_buffer('lower_bounds', lower_bounds)
        self.register_buffer('upper_bounds', upper_bounds)
        self.register_buffer('feature_mask', feature_mask)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Para no modificar el original
        X_trimmed = X.clone()

        mask = self.feature_mask

        X_trimmed[:, mask] = torch.clamp(
            X_trimmed[:, mask],
            self.lower_bounds,
            self.upper_bounds
        )

        return X_trimmed

# Proceso principal (carga de datos)

In [15]:
# Convertir a tensores
x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
x_val_tensor = torch.tensor(x_val.values, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

# Masking para ignorar Latitude y Longitude
features_to_process = [0, 1, 2, 3, 4, 5]
feature_mask = torch.tensor([True if i in features_to_process else False for i in range(8)])
